In [1]:
from collections import Counter

# -----------------------------
# Training Data
# -----------------------------

words = {
    "hug": 2,
    "hugs": 1,
    "pug": 1
}

# Initial WordPiece splits
splits = {
    "hug": ["h", "##u", "##g"],
    "hugs": ["h", "##u", "##g", "##s"],
    "pug": ["p", "##u", "##g"]
}

# Initial vocabulary
vocab = [
    "h",
    "p",
    "##u",
    "##g",
    "##s"
]

print("Initial Vocabulary:")
print(vocab)

print("\nTraining Words:")
print(words)

print("\nInitial Splits:")
print(splits)


# -----------------------------
# Count Token and Pair Frequencies
# -----------------------------

token_freq = Counter()
pair_freq = Counter()

for word, freq in words.items():
    tokens = splits[word]

    # Count individual tokens
    for token in tokens:
        token_freq[token] += freq

    # Count adjacent token pairs
    for i in range(len(tokens) - 1):
        pair = (tokens[i], tokens[i + 1])
        pair_freq[pair] += freq


print("\nToken Frequencies:")
print(token_freq)

print("\nPair Frequencies:")
print(pair_freq)


# -----------------------------
# Calculate WordPiece Scores
# -----------------------------

scores = {}

for pair, freq in pair_freq.items():

    first = pair[0]
    second = pair[1]

    score = freq / (
        token_freq[first] * token_freq[second]
    )

    scores[pair] = score


print("\nWordPiece Scores:")

for pair, score in scores.items():
    print(pair, "=", round(score, 4))


# -----------------------------
# Find Best Pair
# -----------------------------

best_pair = max(
    scores,
    key=scores.get
)

print("\nBest Pair:")
print(best_pair)

print("Highest Score:")
print(round(scores[best_pair], 4))


# -----------------------------
# Merge Best Pair
# -----------------------------

def merge_pair(splits, pair):

    new_token = pair[0] + pair[1].replace("##", "")

    for word in splits:

        tokens = splits[word]
        new_tokens = []

        i = 0

        while i < len(tokens):

            # Check whether current and next token form the best pair
            if (
                i < len(tokens) - 1
                and (tokens[i], tokens[i + 1]) == pair
            ):
                new_tokens.append(new_token)
                i += 2

            else:
                new_tokens.append(tokens[i])
                i += 1

        splits[word] = new_tokens

    return new_token


new_token = merge_pair(splits, best_pair)

vocab.append(new_token)

print("\nNew Token:")
print(new_token)

print("\nUpdated Vocabulary:")
print(vocab)

print("\nUpdated Splits:")
print(splits)


# -----------------------------
# WordPiece Tokenization
# -----------------------------

vocab_set = {
    "h",
    "p",
    "##u",
    "##g",
    "##s",
    "hu",
    "hug",
    "##gs"
}


def tokenize_word(word, vocab):

    tokens = []

    while len(word) > 0:

        found = False

        # Try the longest possible token first
        for i in range(len(word), 0, -1):

            part = word[:i]

            if len(tokens) > 0:
                part = "##" + part

            if part in vocab:

                tokens.append(part)
                word = word[i:]

                found = True
                break

        if not found:
            return ["[UNK]"]

    return tokens


# Tokenize "hugs"
tokens = tokenize_word("hugs", vocab_set)

print("\nWord:")
print("hugs")

print("Tokens:")
print(tokens)


# -----------------------------
# Convert Tokens to Token IDs
# -----------------------------

vocab = {
    "[UNK]": 0,
    "h": 1,
    "p": 2,
    "##u": 3,
    "##g": 4,
    "##s": 5,
    "hu": 6,
    "hug": 7,
    "##gs": 8
}

tokens = ["hug", "##s"]

ids = [
    vocab[token]
    for token in tokens
]

print("\nTokens:")
print(tokens)

print("Token IDs:")
print(ids)


# -----------------------------
# Unknown Word Example
# -----------------------------

unknown_tokens = tokenize_word(
    "bum",
    set(vocab.keys())
)

print("\nUnknown Word:")
print("bum")

print("Tokens:")
print(unknown_tokens)

Initial Vocabulary:
['h', 'p', '##u', '##g', '##s']

Training Words:
{'hug': 2, 'hugs': 1, 'pug': 1}

Initial Splits:
{'hug': ['h', '##u', '##g'], 'hugs': ['h', '##u', '##g', '##s'], 'pug': ['p', '##u', '##g']}

Token Frequencies:
Counter({'##u': 4, '##g': 4, 'h': 3, '##s': 1, 'p': 1})

Pair Frequencies:
Counter({('##u', '##g'): 4, ('h', '##u'): 3, ('##g', '##s'): 1, ('p', '##u'): 1})

WordPiece Scores:
('h', '##u') = 0.25
('##u', '##g') = 0.25
('##g', '##s') = 0.25
('p', '##u') = 0.25

Best Pair:
('h', '##u')
Highest Score:
0.25

New Token:
hu

Updated Vocabulary:
['h', 'p', '##u', '##g', '##s', 'hu']

Updated Splits:
{'hug': ['hu', '##g'], 'hugs': ['hu', '##g', '##s'], 'pug': ['p', '##u', '##g']}

Word:
hugs
Tokens:
['hug', '##s']

Tokens:
['hug', '##s']
Token IDs:
[7, 5]

Unknown Word:
bum
Tokens:
['[UNK]']
